# Dataset candidat bootstrap

Ce notebook charge le dataset candidat MVP produit par le solver. Ce n'est pas un dataset GTO et ce n'est pas un label final: `bootstrap_label` sert seulement a tester le pipeline ML bootstrap.

In [17]:
from pathlib import Path
import csv
import json
import math
from collections import Counter, defaultdict

DATASET_PATH = Path("outputs/bootstrap_candidate_dataset/candidates.csv")

def parse_float(value):
    if value in (None, "", "null"):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def parse_bool(value):
    return str(value).strip().lower() in {"true", "1", "yes"}

rows = []
with DATASET_PATH.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    for raw in reader:
        row = dict(raw)
        for col in ["pot", "to_call", "stack", "spr", "dominant_action_frequency"]:
            row[col] = parse_float(row.get(col))
        row["excluded"] = parse_bool(row.get("excluded"))
        row["action_frequencies"] = json.loads(row.get("action_frequencies") or "{}")
        rows.append(row)

kept_rows = [row for row in rows if not row["excluded"]]
excluded_rows = [row for row in rows if row["excluded"]]

print(f"Dataset: {DATASET_PATH}")
print(f"Lignes totales: {len(rows)}")
print(f"Candidats gardes: {len(kept_rows)}")
print(f"Candidats exclus: {len(excluded_rows)}")
print("Labels gardes:", dict(Counter(row["bootstrap_label"] for row in kept_rows)))
print("Raisons exclusion:", dict(Counter(row["exclusion_reason"] for row in excluded_rows)))

print("\nApercu 3 premieres lignes:")
print(json.dumps(rows[:3], ensure_ascii=False, indent=2))

Dataset: outputs\bootstrap_candidate_dataset\candidates.csv
Lignes totales: 30
Candidats gardes: 12
Candidats exclus: 18
Labels gardes: {'CHECK': 3, 'FOLD': 9}
Raisons exclusion: {'all_in_excluded': 18}

Apercu 3 premieres lignes:
[
  {
    "source_id": "sensitivity_hero_oop_check_or_bet_high_spr_small_call_b33_it25",
    "street": "RIVER",
    "hero_cards": "null",
    "villain_hand": "null",
    "board_cards": "null",
    "pot": 100.0,
    "to_call": 0.0,
    "stack": 2000.0,
    "spr": 20.0,
    "position_model": "OOP",
    "decision_context_type": "hero_check_or_bet",
    "action_frequencies": {
      "ALL_IN": 0.000215,
      "BET_33": 0.022437,
      "CHECK": 0.977348
    },
    "dominant_action": "CHECK",
    "dominant_action_frequency": 0.977348,
    "bootstrap_label": "CHECK",
    "label_source": "solver_candidate",
    "label_quality": "bootstrap_solver_untrusted",
    "excluded": false,
    "exclusion_reason": ""
  },
  {
    "source_id": "sensitivity_hero_oop_check_or_bet_h

## Analyse de donnees

Les correlations ci-dessous sont exploratoires. Le dataset actuel est tres petit, donc les valeurs fortes indiquent surtout des dependances dans l'echantillon MVP, pas une verite poker stable.

In [18]:
def pearson(xs, ys):
    pairs = [(x, y) for x, y in zip(xs, ys) if x is not None and y is not None]
    if len(pairs) < 2:
        return None
    mean_x = sum(x for x, _ in pairs) / len(pairs)
    mean_y = sum(y for _, y in pairs) / len(pairs)
    var_x = sum((x - mean_x) ** 2 for x, _ in pairs)
    var_y = sum((y - mean_y) ** 2 for _, y in pairs)
    if var_x == 0 or var_y == 0:
        return None
    cov = sum((x - mean_x) * (y - mean_y) for x, y in pairs)
    return cov / math.sqrt(var_x * var_y)

def cramers_v(data, feature, target):
    table = defaultdict(Counter)
    for row in data:
        table[row.get(feature)][row.get(target)] += 1
    row_keys = list(table)
    col_keys = sorted({key for counts in table.values() for key in counts})
    n = sum(sum(counts.values()) for counts in table.values())
    if n == 0 or len(row_keys) < 2 or len(col_keys) < 2:
        return None, table
    row_totals = {key: sum(table[key].values()) for key in row_keys}
    col_totals = {key: sum(table[row_key][key] for row_key in row_keys) for key in col_keys}
    chi2 = 0.0
    for row_key in row_keys:
        for col_key in col_keys:
            expected = row_totals[row_key] * col_totals[col_key] / n
            if expected:
                chi2 += (table[row_key][col_key] - expected) ** 2 / expected
    return math.sqrt(chi2 / (n * min(len(row_keys) - 1, len(col_keys) - 1))), table

def print_table(title, table, columns):
    print("\n" + title)
    if not table:
        print("(vide)")
        return
    widths = {col: max(len(col), *(len(str(row.get(col, ""))) for row in table)) for col in columns}
    print(" | ".join(col.ljust(widths[col]) for col in columns))
    print("-+-".join("-" * widths[col] for col in columns))
    for row in table:
        print(" | ".join(str(row.get(col, "")).ljust(widths[col]) for col in columns))

frequency_actions = sorted({action for row in rows for action in row["action_frequencies"]})
for row in rows:
    for action in frequency_actions:
        row[f"action_freq_{action}"] = float(row["action_frequencies"].get(action, 0.0))

numeric_features = [
    "pot",
    "to_call",
    "stack",
    "spr",
    "dominant_action_frequency",
] + [f"action_freq_{action}" for action in frequency_actions]

# Target binaire actuel: dans ce petit export, les labels gardes sont CHECK ou FOLD.
target_is_fold = [1.0 if row["bootstrap_label"] == "FOLD" else 0.0 for row in kept_rows]
corr_to_target = []
for feature in numeric_features:
    corr = pearson([row.get(feature) for row in kept_rows], target_is_fold)
    if corr is not None:
        corr_to_target.append({"feature": feature, "corr_vs_is_FOLD": round(corr, 3)})
corr_to_target.sort(key=lambda row: abs(row["corr_vs_is_FOLD"]), reverse=True)

target_is_excluded = [1.0 if row["excluded"] else 0.0 for row in rows]
corr_to_exclusion = []
for feature in numeric_features:
    corr = pearson([row.get(feature) for row in rows], target_is_excluded)
    if corr is not None:
        corr_to_exclusion.append({"feature": feature, "corr_vs_excluded": round(corr, 3)})
corr_to_exclusion.sort(key=lambda row: abs(row["corr_vs_excluded"]), reverse=True)

categorical_rows = []
for feature in ["position_model", "decision_context_type"]:
    value, table = cramers_v(kept_rows, feature, "bootstrap_label")
    categorical_rows.append({
        "feature": feature,
        "cramers_v_vs_label": None if value is None else round(value, 3),
        "contingency": {key: dict(counts) for key, counts in table.items()},
    })

print_table("Correlations numeriques vs target is_FOLD (candidats gardes)", corr_to_target, ["feature", "corr_vs_is_FOLD"])
print_table("Correlations numeriques vs exclusion", corr_to_exclusion, ["feature", "corr_vs_excluded"])
print_table("Associations categorielles vs bootstrap_label", categorical_rows, ["feature", "cramers_v_vs_label", "contingency"])


Correlations numeriques vs target is_FOLD (candidats gardes)
feature                   | corr_vs_is_FOLD
--------------------------+----------------
action_freq_CHECK         | -1.0           
action_freq_FOLD          | 0.991          
action_freq_BET_33        | -0.606         
to_call                   | 0.587          
pot                       | 0.577          
stack                     | -0.577         
spr                       | -0.577         
action_freq_CALL          | 0.202          
action_freq_RAISE_33      | 0.202          
dominant_action_frequency | -0.14          
action_freq_ALL_IN        | 0.073          

Correlations numeriques vs exclusion
feature                   | corr_vs_excluded
--------------------------+-----------------
action_freq_ALL_IN        | 0.977           
action_freq_FOLD          | -0.799          
dominant_action_frequency | -0.576          
to_call                   | -0.54           
action_freq_CHECK         | -0.408          
action_freq_R

## Resume actuel de l'export

- 30 lignes analysees
- 12 candidats gardes
- 18 candidats exclus, tous avec `all_in_excluded`
- Labels gardes: `FOLD` x9, `CHECK` x3
- Les plus fortes correlations actuelles viennent surtout des frequences d'action solver (`action_freq_CHECK`, `action_freq_FOLD`, `action_freq_ALL_IN`), ce qui est attendu sur un petit export candidat.

## Entrainement LightGBM

Pas d'entrainement lance ici. Cette cellule prepare seulement `X` et `y` pour une etape bootstrap separee.

In [19]:
feature_columns = [
    "pot",
    "to_call",
    "stack",
    "spr",
    "dominant_action_frequency",
] + [f"action_freq_{action}" for action in frequency_actions]

X = [[row.get(column) or 0.0 for column in feature_columns] for row in kept_rows]
y = [row["bootstrap_label"] for row in kept_rows]

print(f"Samples prets: {len(X)}")
print(f"Features: {len(feature_columns)}")
print("Colonnes:", feature_columns)
print("Distribution y:", dict(Counter(y)))

Samples prets: 12
Features: 14
Colonnes: ['pot', 'to_call', 'stack', 'spr', 'dominant_action_frequency', 'action_freq_ALL_IN', 'action_freq_BET_33', 'action_freq_BET_66', 'action_freq_CALL', 'action_freq_CHECK', 'action_freq_FOLD', 'action_freq_RAISE_33', 'action_freq_RAISE_50', 'action_freq_RAISE_66']
Distribution y: {'CHECK': 3, 'FOLD': 9}
